In [4]:
import pandas as pd

# 1. Загрузка исходного датасета из репозитория
# Предполагается, что блокнот лежит в папке notebooks, а данные в папке data
data_path = '../assets/data/Fish.csv' 
df = pd.read_csv(data_path)

# 2. Вывод базовой информации для составления паспорта
print("=== Типы данных и пропуски ===")
df.info()

print("\n=== Первые 5 строк датасета ===")
display(df.head())

print("\n=== Описательные статистики ===")
display(df.describe().round(2))

=== Типы данных и пропуски ===
<class 'pandas.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    str    
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), str(1)
memory usage: 8.8 KB

=== Первые 5 строк датасета ===


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340



=== Описательные статистики ===


,Weight,Length1,Length2,Length3,Height,Width
count,159.00,159.00,159.00,159.00,159.00,159.00
mean,398.33,26.25,28.42,31.23,8.97,4.42
std,357.98,10.00,10.72,11.61,4.29,1.69
min,0.00,7.50,8.40,8.80,1.73,1.05
25%,120.00,19.05,21.00,23.15,5.94,3.39
50%,273.00,25.20,27.30,29.40,7.79,4.25
75%,650.00,32.70,35.50,39.65,12.37,5.58
max,1650.00,59.00,63.40,68.00,18.96,8.14


### Шаг 2. Паспорт данных

| Поле | Тип данных | Единицы / формат | Назначение | Допустимость пропусков | Диапазон или словарь | Источник и способ получения |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Species** | категориальный (`object`) | текст | Признак (вид рыбы) | Нет | Bream, Roach, Whitefish, Parkki, Perch, Pike, Smelt | Визуальное определение / ручной ввод |
| **Weight** | числовой (`float64`) | граммы (г) | Целевая переменная | Нет | > 0 | Взвешивание на весах |
| **Length1** | числовой (`float64`) | см | Признак (стандартная длина) | Нет | > 0; должно быть <= Length2 | Измерение линейкой/лентой |
| **Length2** | числовой (`float64`) | см | Признак (длина до развилки) | Нет | > 0; должно быть >= Length1 | Измерение линейкой/лентой |
| **Length3** | числовой (`float64`) | см | Признак (полная длина) | Нет | > 0; должно быть >= Length2 | Измерение линейкой/лентой |
| **Height** | числовой (`float64`) | см | Признак (максимальная высота) | Нет | > 0 | Измерение линейкой/штангенциркулем |
| **Width** | числовой (`float64`) | см | Признак (максимальная ширина) | Нет | > 0 | Измерение линейкой/штангенциркулем |

#### 2.1 Проверка физического смысла и логических ограничений
1. **Положительность значений**: Ни одно из измерений длины (`Length1`, `Length2`, `Length3`), высоты (`Height`), ширины (`Width`) или массы (`Weight`) не может быть $\le 0$.
2. **Геометрическая согласованность**: Полная длина (`Length3`) всегда должна быть больше или равна длине до развилки хвоста (`Length2`), которая, в свою очередь, должна быть больше или равна стандартной длине (`Length1`): $\text{Length1} \le \text{Length2} \le \text{Length3}$.
3. **Корректность категорий**: Названия видов (`Species`) не должны содержать скрытых дубликатов (пробелов, опечаток, разницы в регистре).

#### 2.2 Единица наблюдения и связь между объектами
* **Единица независимого наблюдения**: Отдельная особь рыбы, извлеченная из улова или выборки.
* **Анализ группировки и временных меток**: 
  * В датасете **отсутствуют** идентификаторы партий, ферм , водоёмов или временные метки.
  * **Вывод**: Ввиду отсутствия группирующих полей данные рассматриваются как независимые и одинаково распределенные. Однако существует риск, что рыбы одного вида были выловлены из одной партии/садка, что необходимо учитывать при выборе стратегии валидации (для исключения смещения выборок будем использовать **стратифицированное разделение по видам**.